In [1]:
import os
import chromadb
from chromadb.utils.embedding_functions import GoogleGeminiEmbeddingFunction
from dotenv import load_dotenv
from scripts.helpers import MODEL
from pandas import read_csv
from scripts.helpers import count_tokens, estimate_cost, create_movie_text, create_movie_metadatas
import json

In [2]:
# Create a persistant client
client = chromadb.PersistentClient(path="./chroma_db")
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

## Read the netflix_titles.csv file

In [3]:
# Read the netflix_titles.csv file
netflix_titles_csv = read_csv("./netflix_titles.csv")
netflix_titles_dict = netflix_titles_csv.to_dict(orient="records")[0:100]
netflix_titles_ids = netflix_titles_csv.to_dict(orient="list")['show_id'][0:100]
documents = [create_movie_text(movie) for movie in netflix_titles_dict]
metadatas = create_movie_metadatas(netflix_titles_dict)
print(documents[0], len(documents))


    Title: Dick Johnson Is Dead
    Description: As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.
    Categories: Documentaries
     100


## Estimating costs

In [4]:
estimated_result = estimate_cost(api_key, documents)
print("Total tokens:", estimated_result["total_tokens"])
print("Cost:", estimated_result["cost_usd"])

Total tokens: 5779
Cost: 0.0008668499999999998


## Adding data to the collection

Create a netflix_title collection using the OpenAI Embedding function.

Just for native OpenAI API key

```python
collection = client.create_collection(
    name="netflix_titles",
    embedding_function=OpenAIEmbeddingFunction(model_name=MODEL, api_key="<OPENAI_API_TOKEN>") 
)
print(client.list_collections())
```

In [5]:
# Create a netflix_title collection using the Gemini Embedding function
COLLECTION_NAME = "netflix_titles"

# GoogleGeminiEmbeddingFunction no recibe la api_key: recibe el NOMBRE de la
# variable de entorno donde buscarla (default "GEMINI_API_KEY"), y la lee con
# os.getenv al instanciarse. load_dotenv() de la celda anterior ya la puso ahi.
google_ef = GoogleGeminiEmbeddingFunction(model_name=MODEL)

# get_or_create: la trae si ya existe, o la crea si no
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=google_ef
)
print(client.list_collections())

[Collection(name=netflix_titles)]


In [6]:
collection.add(ids=netflix_titles_ids, documents=documents, metadatas=metadatas)
# Print the collection size and first ten items
print(f"No. of documents: {collection.count()}")
print(f"First ten documents: {collection.peek()}")

No. of documents: 100
First ten documents: {'ids': ['s1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10'], 'embeddings': array([[-0.02376537,  0.0038919 , -0.00075665, ..., -0.01339986,
         0.00875239, -0.00127007],
       [-0.01185326, -0.00555816,  0.024309  , ..., -0.00596455,
        -0.01414619, -0.00356004],
       [-0.01163226, -0.00623306,  0.03517089, ...,  0.00109072,
        -0.00768444, -0.01039738],
       ...,
       [-0.01678621, -0.01001198,  0.0339636 , ...,  0.00279733,
         0.02489963,  0.00645387],
       [-0.0402812 , -0.01114775,  0.03133426, ...,  0.01951658,
        -0.00296695,  0.00641462],
       [-0.01892494,  0.00426874,  0.02301583, ...,  0.007902  ,
         0.00806093, -0.01009896]], shape=(10, 3072)), 'documents': ['\n    Title: Dick Johnson Is Dead\n    Description: As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.\n    Categories: Doc

In [7]:
print(json.dumps(collection.get(ids=["s1"], include=["metadatas", "documents"]), indent=2))


{
  "ids": [
    "s1"
  ],
  "embeddings": null,
  "documents": [
    "\n    Title: Dick Johnson Is Dead\n    Description: As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.\n    Categories: Documentaries\n    "
  ],
  "uris": null,
  "included": [
    "metadatas",
    "documents"
  ],
  "data": null,
  "metadatas": [
    {
      "rating": "PG-13",
      "type": "Movie",
      "duration": "90 min",
      "release_year": 2020,
      "title": "Dick Johnson Is Dead",
      "country": "United States"
    }
  ]
}


## Querying and updating the database

In [8]:
result = collection.query(
  query_texts=["films about dogs"],
  n_results=3
)

print(json.dumps(result, indent=2))

{
  "ids": [
    [
      "s95",
      "s46",
      "s10"
    ]
  ],
  "embeddings": null,
  "documents": [
    [
      "\n    Title: Show Dogs\n    Description: A rough and tough police dog must go undercover with an FBI agent as a prim and proper pet at a dog show to save a baby panda from an illegal sale.\n    Categories: Children & Family Movies, Comedies\n    ",
      "\n    Title: My Heroes Were Cowboys\n    Description: Robin Wiltshire's painful childhood was rescued by Westerns. Now he lives on the frontier of his dreams, training the horses he loves for the big screen.\n    Categories: Documentaries\n    ",
      "\n    Title: The Starling\n    Description: A woman adjusting to life after a loss contends with a feisty bird that's taken over her garden \u2014 and a husband who's struggling to find a way forward.\n    Categories: Comedies, Dramas\n    "
    ]
  ],
  "uris": null,
  "included": [
    "metadatas",
    "documents",
    "distances"
  ],
  "data": null,
  "metadatas":

## Multiple queries

In [9]:
reference_ids = ['s99', 's100']

# Retrieve the documents for the reference_ids
reference_texts = collection.get(ids=reference_ids)['documents']

# Query using reference_texts
result = collection.query(
    query_texts=reference_texts,
    n_results=3
)

print(json.dumps(result['documents'], indent=2))

[
  [
    "\n    Title: Octonauts: Above & Beyond\n    Description: The Octonauts expand their exploration beyond the sea \u2014\u00a0and onto land! With new rides and new friends, they'll protect any habitats and animals at risk.\n    Categories: British TV Shows, Kids' TV\n    ",
    "\n    Title: A StoryBots Space Adventure\n    Description: Join the StoryBots and the space travelers of the historic Inspiration4 mission as they search for answers to kids' questions about space.\n    Categories: Children & Family Movies\n    ",
    "\n    Title: Pok\u00e9mon Master Journeys: The Series\n    Description: As Ash battles his way through the World Coronation Series, Goh continues his quest to catch every Pok\u00e9mon. Together, they're on a journey to adventure!\n    Categories: Anime Series, Kids' TV\n    "
  ],
  [
    "\n    Title: On the Verge\n    Description: Four women \u2014 a chef, a single mom, an heiress and a job seeker \u2014 dig into love and work, with a generous side of m

## Filtering by metadata

Updating the collection with metadatas. This is another query, so instead of update the collection let's to add metadatas at the same time we create the collection

In [ ]:
# collection.update(ids=netflix_titles_ids, metadatas=create_movie_metadatas(netflix_titles_dict))

In [10]:
reference_texts = ["children's story about a car", "lions"]

# Query two results using reference_texts
result = collection.query(
    query_texts=reference_texts,
    n_results=2,
    # Filter for titles with a G rating released before 2019
    where={
    "$and": [
            {"rating": 
                {"$eq": "G"}
            },
            {"release_year": 
                {"$lt": 2019}
            }
        ]
    }
)

print(result['documents'])

ValueError: Failed to generate embeddings: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 20.299223922s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'gl